In [ ]:
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input 
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import numpy as np

I0000 00:00:1789545405.484342   54177 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789545405.547412   54177 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789545407.217514   54177 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
training_path = "../Datasets/data/train/Medical/X-Ray"
testing_path = "../Datasets/data/test/Medical/X-Ray"
valid_path = "../Datasets/data/vaild/Medical/X-Ray"

In [3]:
train_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 10,
    zoom_range = 0.2,
    horizontal_flip = True
)
test_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)
valid_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

In [4]:
training_set = train_dataug.flow_from_directory(
    training_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical"
)
valid_set = valid_dataug.flow_from_directory(
    valid_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical"
)
testing_set = test_dataug.flow_from_directory(
    testing_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "categorical",
    shuffle = False
)

Found 27764 images belonging to 6 classes.
Found 3533 images belonging to 6 classes.
Found 5615 images belonging to 6 classes.


In [5]:
testing_set.class_indices

{'chest x-ray': 0,
 'elbow': 1,
 'forearm': 2,
 'hand': 3,
 'humerus': 4,
 'shoulder': 5}

In [16]:
base_model = EfficientNetV2B0(weights="imagenet",include_top=False,input_shape=(224,224,3))

In [17]:
base_model.trainable = False

In [18]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128,activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
output = Dense(6,activation="sigmoid")(x)

In [19]:
model = Model(inputs = base_model.input,outputs = output)

In [20]:
model.compile(optimizer=Adam(learning_rate=0.002),metrics=["accuracy"],loss="categorical_crossentropy")

In [21]:
early_stop = EarlyStopping(monitor="loss",patience=3,restore_best_weights=True)

In [22]:
model.fit(training_set,validation_data=(valid_set),epochs=15,callbacks=[early_stop])

Epoch 1/4


I0000 00:00:1789110153.296118   20279 service.cc:153] XLA service 0x74ddb0047b40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789110153.296530   20279 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1789110153.814126   20279 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789110156.425618   20279 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1789110157.150689   20279 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_25745__.271
I0000 00:00:1789110181.676844   20279 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 875/1736 ━━━━━━━━━━━━━━━━━━━━ 3:04 214ms/step - accuracy: 0.8181 - loss: 0.5004

I0000 00:00:1789110370.105863   20280 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_25745__.271


1736/1736 ━━━━━━━━━━━━━━━━━━━━ 472s 249ms/step - accuracy: 0.9038 - loss: 0.2749 - val_accuracy: 0.9527 - val_loss: 0.1419
Epoch 2/4
1736/1736 ━━━━━━━━━━━━━━━━━━━━ 377s 217ms/step - accuracy: 0.9420 - loss: 0.1732 - val_accuracy: 0.9691 - val_loss: 0.1047
Epoch 3/4
1736/1736 ━━━━━━━━━━━━━━━━━━━━ 307s 177ms/step - accuracy: 0.9393 - loss: 0.1617 - val_accuracy: 0.9629 - val_loss: 0.1186
Epoch 4/4
1736/1736 ━━━━━━━━━━━━━━━━━━━━ 312s 180ms/step - accuracy: 0.9493 - loss: 0.1416 - val_accuracy: 0.9711 - val_loss: 0.1211


In [23]:
prediction = model.predict(testing_set)

351/351 ━━━━━━━━━━━━━━━━━━━━ 48s 119ms/step


In [24]:
true_prediction = np.argmax(prediction,axis=1)

In [25]:
true_prediction

array([0, 0, 0, ..., 5, 5, 5], shape=(5615,))

In [26]:
y_true = testing_set.classes

In [27]:
print(classification_report(y_true,true_prediction))
confusion_matrix(y_true,true_prediction)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1550
           1       0.86      0.99      0.92       811
           2       0.94      0.81      0.87       461
           3       0.99      1.00      0.99       713
           4       0.96      0.79      0.86       538
           5       0.97      0.98      0.98      1542

    accuracy                           0.96      5615
   macro avg       0.95      0.93      0.94      5615
weighted avg       0.96      0.96      0.96      5615



array([[1550,    0,    0,    0,    0,    0],
       [   0,  805,    4,    0,    0,    2],
       [   0,   72,  375,    8,    3,    3],
       [   0,    1,    2,  710,    0,    0],
       [   0,   53,   20,    2,  423,   40],
       [   3,    8,    0,    0,   15, 1516]])

In [30]:
model.save("X_ray_category_model.keras")